In [3]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost.core import XGBoostError
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

print(f"XGBoost version: {xgb.__version__}")

XGBoost version: 3.2.0


In [2]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 1.9 MB/s eta 0:00:55
    --------------------------------------- 1.6/101.7 MB 3.6 MB/s eta 0:00:28
   - -------------------------------------- 3.9/101.7 MB 6.5 MB/s eta 0:00:15
   - -------------------------------------- 5.0/101.7 MB 7.5 MB/s eta 0:00:13
   -- ------------------------------------- 5.2/101.7 MB 5.2 MB/s eta 0:00:19
   -- ------------------------------------- 5.5/101.7 MB 4.7 MB/s eta 0:00:21
   -- ------------------------------------- 5.8/101.7 MB 4.3 MB/s eta 0:00:23
   -- ------------------------------------- 6.8/101.7 MB 4.1 MB/s eta 0:00:24
   -- ------------------------------------- 7.1/101.7 MB 4.0 MB/s eta 0:00:24
   -- ------------------------------------- 7.1/101.7 MB 4.0 MB/s eta 0:00:24
   --- 

In [4]:
feature_columns = [
    "Study_Hours",
    "Sleep_Hours",
    "Practice_Tests",
    "Classes_Attended",
]
target_column = "Final_Score"

data_path = Path("linear models") / "student_data.csv"
if not data_path.exists():
    data_path = Path("..") / "linear models" / "student_data.csv"
if not data_path.exists():
    raise FileNotFoundError(
        "Could not find 'linear models/student_data.csv'. Run this notebook "
        "from the repository root or the 'xg boost' folder."
    )

data = pd.read_csv(data_path)
required_columns = feature_columns + [target_column]
missing_columns = [column for column in required_columns if column not in data.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

X = data[feature_columns]
y = data[target_column]
print(f"Loaded {len(data)} rows from {data_path}")
data.head()

Loaded 500 rows from ..\linear models\student_data.csv


,Study_Hours,Sleep_Hours,Practice_Tests,Classes_Attended,Final_Score
0,4.370861,7.490809,3,28,87.157478
1,9.556429,6.680482,5,13,98.991520
2,7.587945,5.547638,2,7,83.344530
3,6.387926,8.068975,4,14,85.622403
4,2.404168,7.423656,0,23,61.122435


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

Training rows: 400
Test rows: 100


In [6]:
models = {}
benchmark_rows = []

for mode, device in [("CPU", "cpu"), ("GPU", "cuda")]:
    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        tree_method="hist",
        device=device,
        random_state=42,
    )

    try:
        fit_start = perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = perf_counter() - fit_start

        prediction_start = perf_counter()
        predictions = model.predict(X_test)
        prediction_seconds = perf_counter() - prediction_start

        models[mode] = model
        benchmark_rows.append(
            {
                "Mode": mode,
                "Fit time (seconds)": fit_seconds,
                "Prediction time (seconds)": prediction_seconds,
                "Total time (seconds)": fit_seconds + prediction_seconds,
                "MAE": mean_absolute_error(y_test, predictions),
                "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
                "R2": r2_score(y_test, predictions),
            }
        )
        print(f"{mode} training and evaluation completed.")
    except XGBoostError as error:
        if mode == "CPU":
            raise
        print(f"GPU run could not be completed; the CPU results are still available.\n{error}")

CPU training and evaluation completed.
GPU training and evaluation completed.


d:\ANN_PTROJECT\penve\Lib\site-packages\xgboost\core.py:751: UserWarning: [13:36:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [7]:
if not benchmark_rows:
    raise RuntimeError("The CPU benchmark did not produce a result.")

benchmark = pd.DataFrame(benchmark_rows).set_index("Mode")
display(benchmark)

prediction_mode = "GPU" if "GPU" in models else "CPU"
prediction_model = models[prediction_mode]
print(f"Manual predictions will use the {prediction_mode} model.")

,Fit time (seconds),Prediction time (seconds),Total time (seconds),MAE,RMSE,R2
Mode,,,,,,
CPU,0.329396,0.003414,0.332810,4.289739,5.464130,0.851463
GPU,0.477442,0.021035,0.498477,4.289745,5.464135,0.851462


Manual predictions will use the GPU model.


In [9]:
print("Enter values for the following features to predict a student's final score.")
manual_values = {}
for feature in feature_columns:
    while True:
        try:
            manual_values[feature] = float(input(f"{feature}: "))
            break
        except ValueError:
            print("Please enter a valid number.")

manual_input = pd.DataFrame([manual_values], columns=feature_columns)
predicted_score = prediction_model.predict(manual_input)[0]
print(f"Predicted Final_Score: {predicted_score:.2f}")

Enter values for the following features to predict a student's final score.
Predicted Final_Score: 87.54
